In [ ]:
!pip install kagglehub -q
import kagglehub

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Dataset downloaded to: /kaggle/input/telco-customer-churn


In [ ]:
import pandas as pd
import os

# Find the actual CSV file in the downloaded folder
for file in os.listdir(path):
    print(file)

WA_Fn-UseC_-Telco-Customer-Churn.csv


In [ ]:
df = pd.read_csv(f"{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:")
df.head()

Shape: (7043, 21)

Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
print("Churn distribution:")
print(df['Churn'].value_counts())
print("\nChurn rate:", round(df['Churn'].value_counts(normalize=True)['Yes'] * 100, 1), "%")

print("\nMissing values:")
print(df.isnull().sum().sum(), "total missing values")

print("\nData types:")
print(df.dtypes)

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.5 %

Missing values:
0 total missing values

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [ ]:
# Check for the hidden issue in TotalCharges
print(df[df['TotalCharges'] == ' ']['TotalCharges'].count(), "rows with blank TotalCharges")

# Convert to numeric, forcing errors to NaN so we can see them
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("\nMissing after conversion:", df['TotalCharges'].isnull().sum())

# These are likely brand new customers (tenure = 0)
print("\nTenure for these rows:")
print(df[df['TotalCharges'].isnull()]['tenure'].unique())


11 rows with blank TotalCharges

Missing after conversion: 11

Tenure for these rows:
[0]


In [ ]:
# Since these are brand new customers (tenure=0), TotalCharges should logically be 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("Missing values now:", df['TotalCharges'].isnull().sum())
print("\nFinal check - data types:")
print(df.dtypes)

Missing values now: 0

Final check - data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


In [ ]:
import matplotlib.pyplot as plt

# Churn rate by contract type - usually the biggest driver
churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
print("Churn rate by contract type:")
print(churn_by_contract.round(1))

# Churn rate by tenure groups
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-12mo', '13-24mo', '25-48mo', '49-72mo'])
churn_by_tenure = df.groupby('tenure_group', observed=True)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
print("\nChurn rate by tenure group:")
print(churn_by_tenure.round(1))

# Average monthly charges: churned vs retained
print("\nAvg monthly charges:")
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))

Churn rate by contract type:
Contract
Month-to-month    42.7
One year          11.3
Two year           2.8
Name: Churn, dtype: float64

Churn rate by tenure group:
tenure_group
0-12mo     47.7
13-24mo    28.7
25-48mo    20.4
49-72mo     9.5
Name: Churn, dtype: float64

Avg monthly charges:
Churn
No     61.27
Yes    74.44
Name: MonthlyCharges, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Prepare features - drop customerID (not predictive) and tenure_group (we'll use raw tenure)
model_df = df.drop(['customerID', 'tenure_group'], axis=1)

# Encode all categorical columns
label_encoders = {}
for col in model_df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col])
    label_encoders[col] = le

# Split features and target
X = model_df.drop('Churn', axis=1)
y = model_df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (5634, 19)
Test set: (1409, 19)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_pred_proba), 3))


              precision    recall  f1-score   support

    No Churn       0.87      0.80      0.83      1035
       Churn       0.55      0.68      0.61       374

    accuracy                           0.77      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.77      0.77      1409

ROC-AUC Score: 0.838


In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(10))

             feature  importance
14          Contract    0.165906
4             tenure    0.149895
18      TotalCharges    0.134436
17    MonthlyCharges    0.126135
8     OnlineSecurity    0.084046
11       TechSupport    0.065158
7    InternetService    0.048854
16     PaymentMethod    0.044301
9       OnlineBackup    0.030156
15  PaperlessBilling    0.022793


In [ ]:
import pickle

# Save the trained model
with open('churn_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save the label encoders (needed to decode predictions later)
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# Save a clean version of the dataset for Power BI
df.to_csv('telco_churn_cleaned.csv', index=False)

print("Model and cleaned data saved!")

Model and cleaned data saved!


In [ ]:
from google.colab import files

files.download('churn_model.pkl')
files.download('label_encoders.pkl')
files.download('telco_churn_cleaned.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>